# 🌵 Project 29 — Drought Monitoring Using NDVI

This is an important step because you're moving from simply detecting **vegetation change** to asking an environmental question:

> **Did vegetation experience stress during a drought year?**

You've already learned:

- NDVI → vegetation condition
- Time series → vegetation behavior over time
- `.subtract()` → pixel-by-pixel change
- Change maps → increase/decrease
- Multi-year comparison → long-term environmental change

Now we'll combine these ideas to monitor **drought-related vegetation stress**.

---

# 1. Introduce the Project

## Goal

Compare vegetation during a **drought year** with vegetation during **normal years**.

For example:

```text
Normal years
    ↓
Average NDVI
    ↓
Compare with
    ↓
Drought year NDVI
    ↓
Vegetation stress map
````

### Real-world applications

This workflow can be used for:

* 🌾 Agricultural drought monitoring
* 🌳 Forest stress detection
* 🌿 Rangeland monitoring
* 🏜️ Desertification studies
* 🌍 Climate impact analysis
* 💧 Water-resource management

---

# 2. Big Picture

Instead of comparing only:

```text
2020 vs 2025
```

we'll establish a **baseline** from several normal years.

```text
Normal Year 1 ─┐
Normal Year 2 ─┤
Normal Year 3 ─┤
Normal Year 4 ─┘
       ↓
Average NDVI
       ↓
    Baseline
       ↓
Compare with
       ↓
  Drought Year
       ↓
 NDVI Anomaly
       ↓
Vegetation Stress
```

This is more meaningful than comparing a drought year with just one other year.

---

# 3. What Is a Drought Signal in NDVI?

Suppose the normal NDVI for an area is:

```text
Normal NDVI = 0.65
```

But during a drought:

```text
Drought NDVI = 0.40
```

Then:

```text
0.40 - 0.65 = -0.25
```

A negative anomaly suggests vegetation was less vigorous than normal.

We can call this:

> **Negative NDVI anomaly**

---

# 4. Important Scientific Point

A low NDVI does **not automatically prove drought**.

NDVI can decrease because of:

* Drought
* Harvest
* Fire
* Land-use change
* Disease
* Flooding
* Seasonal differences
* Clouds or poor imagery

So NDVI is a **vegetation stress indicator**, not a direct drought measurement.

For real research, you would ideally combine it with rainfall, soil moisture, temperature, or drought indices.

---

# 5. Step 1 — Define the Study Area

We'll use an agricultural region.

```javascript
var studyArea = ee.Geometry.Polygon([
  [
    [51.20,32.40],
    [51.70,32.40],
    [51.70,32.80],
    [51.20,32.80],
    [51.20,32.40]
  ]
]);
```

---

# 6. Step 2 — Load Sentinel-2

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);
```

We're going to compare the **same growing-season period** across multiple years.

For example:

```text
May → August
```

This is important because we don't want seasonal changes to look like drought effects.

---

# 7. Step 3 — Create a Function for Seasonal NDVI

We already created similar functions in previous projects.

```javascript
function getSeasonNDVI(startDate, endDate) {

  var image = collection
    .filterBounds(studyArea)
    .filterDate(startDate, endDate)
    .median();

  return image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');
}
```

The function takes:

```text
Start date
    +
End date
    ↓
Sentinel-2 images
    ↓
Median composite
    ↓
NDVI
```

---

# 8. Step 4 — Create Normal-Year NDVI

Let's say we consider these years relatively normal:

```text
2021
2022
2023
2024
```

We can calculate NDVI for each.

```javascript
var ndvi2021 = getSeasonNDVI(
  '2021-05-01',
  '2021-08-31'
);

var ndvi2022 = getSeasonNDVI(
  '2022-05-01',
  '2022-08-31'
);

var ndvi2023 = getSeasonNDVI(
  '2023-05-01',
  '2023-08-31'
);

var ndvi2024 = getSeasonNDVI(
  '2024-05-01',
  '2024-08-31'
);
```

Now we have four NDVI images.

---

# 9. Step 5 — Create the Baseline ⭐

We want the average vegetation condition across the normal years.

```javascript
var baseline = ee.ImageCollection([
  ndvi2021,
  ndvi2022,
  ndvi2023,
  ndvi2024
]).mean();
```

Conceptually:

```text
2021 NDVI ─┐
2022 NDVI ─┤
2023 NDVI ─┤ → mean() → Baseline NDVI
2024 NDVI ─┘
```

The baseline represents:

> **Typical vegetation condition for this period.**

---

# 10. Step 6 — Calculate the Drought-Year NDVI

Let's use 2025 as our drought-year example.

```javascript
var droughtNDVI = getSeasonNDVI(
  '2025-05-01',
  '2025-08-31'
);
```

Now we have:

```text
Baseline NDVI
       +
Drought-year NDVI
```

---

# 11. Step 7 — Calculate NDVI Anomaly ⭐

Now we perform the important comparison:

```javascript
var ndviAnomaly = droughtNDVI
  .subtract(baseline);
```

This means:

```text
Drought NDVI - Normal NDVI
```

Example:

```text
Normal = 0.70
Drought = 0.45

0.45 - 0.70
     ↓
   -0.25
```

Negative value:

➡️ Vegetation is below its normal condition.

---

# 12. Step 8 — Create a Stress Mask

We can identify areas where NDVI decreased substantially.

```javascript
var stress = ndviAnomaly.lt(-0.1);
```

This means:

> Find pixels where NDVI is at least 0.1 lower than the baseline.

Example:

```text
-0.30 → Stress
-0.15 → Stress
-0.05 → No stress
+0.10 → No stress
```

---

# 13. Step 9 — Visualize the Anomaly

```javascript
var anomalyVis = {
  min: -0.5,
  max: 0.5,
  palette: [
    'red',
    'white',
    'green'
  ]
};
```

Then:

```javascript
Map.centerObject(
  studyArea,
  10
);

Map.addLayer(
  ndviAnomaly,
  anomalyVis,
  'NDVI Anomaly'
);
```

Interpretation:

```text
🔴 Red
Negative anomaly
Vegetation below normal

⚪ White
Near normal

🟢 Green
Positive anomaly
Vegetation above normal
```

---

# 14. Step 10 — Display Drought Stress

```javascript
Map.addLayer(
  stress,
  {
    min: 0,
    max: 1,
    palette: [
      'white',
      'red'
    ]
  },
  'Vegetation Stress'
);
```

Now you have a simple drought-related vegetation stress map.

---

# Complete Code

```javascript
// 1. Study Area
var studyArea = ee.Geometry.Polygon([
  [
    [51.20,32.40],
    [51.70,32.40],
    [51.70,32.80],
    [51.20,32.80],
    [51.20,32.40]
  ]
]);


// 2. Sentinel-2 Collection
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);


// 3. Seasonal NDVI Function
function getSeasonNDVI(startDate, endDate) {

  var image = collection
    .filterBounds(studyArea)
    .filterDate(startDate, endDate)
    .median();

  return image
    .normalizedDifference(['B8','B4'])
    .rename('NDVI');
}


// 4. Normal Years
var ndvi2021 = getSeasonNDVI(
  '2021-05-01',
  '2021-08-31'
);

var ndvi2022 = getSeasonNDVI(
  '2022-05-01',
  '2022-08-31'
);

var ndvi2023 = getSeasonNDVI(
  '2023-05-01',
  '2023-08-31'
);

var ndvi2024 = getSeasonNDVI(
  '2024-05-01',
  '2024-08-31'
);


// 5. Baseline
var baseline = ee.ImageCollection([
  ndvi2021,
  ndvi2022,
  ndvi2023,
  ndvi2024
]).mean();


// 6. Drought Year
var droughtNDVI = getSeasonNDVI(
  '2025-05-01',
  '2025-08-31'
);


// 7. NDVI Anomaly
var ndviAnomaly = droughtNDVI
  .subtract(baseline);


// 8. Vegetation Stress
var stress = ndviAnomaly.lt(-0.1);


// 9. Visualization
var anomalyVis = {
  min: -0.5,
  max: 0.5,
  palette: [
    'red',
    'white',
    'green'
  ]
};


// 10. Display
Map.centerObject(
  studyArea,
  10
);

Map.addLayer(
  ndviAnomaly,
  anomalyVis,
  'NDVI Anomaly'
);

Map.addLayer(
  stress,
  {
    min: 0,
    max: 1,
    palette: [
      'white',
      'red'
    ]
  },
  'Vegetation Stress'
);
```

---

# Things to Remember

## 1. Baseline

A baseline represents the **normal or expected condition**.

```text
Several normal years
        ↓
      Mean
        ↓
     Baseline
```

This is much stronger than comparing a drought year against one arbitrary year.

---

## 2. NDVI Anomaly

The general formula is:

```text
NDVI Anomaly
=
Drought/Target Year NDVI
-
Baseline NDVI
```

Therefore:

```text
Negative → Below normal
Zero     → Near normal
Positive → Above normal
```

---

## 3. `mean()`

You've already learned `.median()`.

Now we're using:

```javascript
.mean()
```

`mean()` calculates the average value across the images.

For example:

```text
0.60
0.70
0.65
0.75

Mean = 0.675
```

---

# Common Beginner Mistakes

❌ **Mistake 1:** Comparing different seasons.

```text
2021 January
```

vs

```text
2025 July
```

This isn't a good drought comparison.

Use the same seasonal window.

---

❌ **Mistake 2:** Using only one "normal" year.

A single year might itself have unusual weather.

Better:

```text
Several years
     ↓
Baseline
```

---

❌ **Mistake 3:** Calling every negative NDVI anomaly "drought."

A negative anomaly means:

> Vegetation was below the baseline.

It **may** be related to drought, but you need additional evidence to confidently attribute the cause to drought.

---

❌ **Mistake 4:** Using unmasked cloudy imagery.

Clouds can artificially reduce or distort NDVI.

For serious analysis, use cloud masking and preferably robust multi-image composites.

---

# Your Practice — Project 29.1

Create your own drought-monitoring analysis.

### Requirements

1. Choose a study area.
2. Choose **at least 3 normal years**.
3. Choose one target/drought year.
4. Use the same growing-season period for every year.
5. Calculate seasonal NDVI.
6. Create a baseline from the normal years.
7. Calculate the target year's NDVI anomaly.
8. Create a vegetation-stress mask.
9. Display the anomaly and stress maps.

### Bonus

Calculate the **mean NDVI of the entire study area** for:

* Each normal year
* The drought year

Then compare them.

---

# Conceptual Questions

### Question 1

Why is a multi-year baseline better than comparing the drought year with only one previous year?

### Question 2

If:

```text
Baseline NDVI = 0.72
```

```text
Drought-year NDVI = 0.48
```

what is the NDVI anomaly?

### Question 3

Does a negative NDVI anomaly **prove** that drought caused the vegetation stress? Why or why not?

```
```
